In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
import itertools
from scipy.stats import pearsonr, spearmanr, linregress

sns.set_theme()
sns.set_context('notebook')


In [ ]:
dataset = Path.joinpath(Path.cwd(), 'dataset')
srp_path = Path.joinpath(dataset, 'survey_results_public.csv')

df = pd.read_csv(srp_path, index_col='ResponseId')
df

In [ ]:
x = np.linspace(0,10,100)
y_random = np.random.uniform(-1,1,100)
y_corr = x*2 + y_random

In [ ]:
fig, (ax_esq, ax_dir) = plt.subplots(figsize=(18, 8), ncols=2)

sns.scatterplot(ax=ax_esq, x=x, y=y_corr)
ax_esq.set_title('Correlação alta')

sns.scatterplot(ax=ax_dir, x=x, y=y_random)
ax_dir.set_title('Sem correlação')

fig.suptitle('Exemplos de correlação')
plt.show()

In [ ]:
x = np.linspace(0, 10, 100)
df_corr = pd.DataFrame({
    'Correlação perfeita (+)': x * 2,
    'Correlação alta (+)': x + np.random.uniform(-1, 1, 100),
    'Correlação baixa (+)': x + np.random.uniform(-10, 10, 100),
    'Correlação perfeita (-)': -x * 2,
    'Correlação alta (-)': -x + np.random.uniform(-1, 1, 100),
    'Correlação baixa (-)': -x + np.random.uniform(-10, 10, 100),
})

fig, axes = plt.subplots(figsize=(18, 8), ncols=3, nrows=2)

for column, ax in zip(df_corr, itertools.chain.from_iterable(axes)):
    y = df_corr[column]
    sns.scatterplot(ax=ax, x=x, y=y)
    ax.set_title(column)

fig.suptitle('Exemplos de correlação')
plt.show()

In [ ]:
for column in df_corr:
    r, _ = pearsonr(x, df_corr[column])
    print(f'A correlação entre X e coluna {column} é {r:.3f}')

In [ ]:
sample = pd.DataFrame({
    'x': [10, 5, 2],
    'y': [9, 8, 9],
    'z': [4, 4, 7],
})
sample

In [ ]:
sample.corr()

In [ ]:
#Gráfico de heatmap ajuda a visualizar melhor correlações

fig, ax = plt.subplots(figsize=(18,8))
sns.heatmap(data=sample.corr(), square=True, cmap='RdBu', annot=True)

fig.suptitle('Mapa de correlação')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))
sns.scatterplot(data=df, ax=ax, x='WorkExp', y='ConvertedCompYearly')

fig.suptitle('Correlação entre anos de trabalho e salário')
plt.show()

In [ ]:
df_filter = df.loc[df['ConvertedCompYearly'] <= 1_000_000].copy()

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))
sns.scatterplot(data=df_filter, ax=ax, x='WorkExp', y='ConvertedCompYearly')

fig.suptitle('Correlação entre anos de trabalho e salário')
plt.show()

In [ ]:
#novo valor de corte, para gerar um gráfico mais voltado para a maioria dos dados
cutoff = 300_000

upper_df = df_filter.loc[df_filter['ConvertedCompYearly'] > cutoff]
lower_df = df_filter.loc[df_filter['ConvertedCompYearly'] <= cutoff]

fig, (ax_esq, ax_dir) = plt.subplots(figsize=(18, 8), ncols=2)

sns.scatterplot(data=upper_df, ax=ax_esq, x='WorkExp', y='ConvertedCompYearly')
ax_esq.set_title('Salário maior')

sns.scatterplot(data=lower_df, ax=ax_dir, x='WorkExp', y='ConvertedCompYearly')
ax_dir.set_title('Salário menor')

fig.suptitle('Correlação entre anos de trabalho e salário')
plt.show()

In [ ]:
print('Correlação no salário menor')
lower_df[['WorkExp', 'ConvertedCompYearly']].corr()

In [ ]:
print('Correlação no salário maior')
upper_df[['WorkExp', 'ConvertedCompYearly']].corr()

In [ ]:
fig, (ax_esq, ax_dir) = plt.subplots(figsize=(18, 8), ncols=2)

cmap = 'viridis'

#discrete (True, False) -> permite o uso de valores discretos no x (inteiros)
sns.histplot(data=upper_df, ax=ax_esq, x='WorkExp', y='ConvertedCompYearly', discrete=(True, False), cmap=cmap, cbar=True)
ax_esq.set_title('Salário maior')

sns.histplot(data=lower_df, ax=ax_dir, x='WorkExp', y='ConvertedCompYearly', discrete=(True, False), cmap=cmap, cbar=True)
ax_dir.set_title('Salário menor')

fig.suptitle('Correlação entre anos de trabalho e salário')
plt.show()

In [ ]:
#Correlação por segmentos

paises = ['United States of America', 'Germany', 'Brazil', 'South Africa', 'India', 'Australia']
df_paises = df_filter.loc[df_filter['Country'].isin(paises)].copy()

In [ ]:
df_paises['Country'].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))
sns.scatterplot(data=df_paises, ax=ax, x='WorkExp', y='ConvertedCompYearly', hue='Country')

fig.suptitle('Correlação entre anos de trabalho e salário')
plt.show()

In [ ]:
#relplot -> permite observar cada gráfico em uma coluna diferente de um plot maior

g = sns.relplot(data=df_paises, x='WorkExp', y='ConvertedCompYearly', col='Country', hue='Country', col_wrap=3)
plt.show()

In [ ]:
df_paises[['WorkExp', 'ConvertedCompYearly', 'Country']].groupby('Country').corr()

In [ ]:
#Filtro com iloc para pegar o valor que interessa na matriz

df_paises[['WorkExp', 'ConvertedCompYearly', 'Country']].groupby('Country').corr().iloc[1::2, 0].sort_values(ascending=False)

In [ ]:
#Pairplot -> permite plotar correlações e distribuições dos dados de uma única vez

df['isBrazil'] = (df['Country'] == 'Brazil').astype(int)
df['LogComp'] = np.log(df['ConvertedCompYearly'])

df_filtered = df.loc[df['ConvertedCompYearly'] <= 300_000][['WorkExp', 'ConvertedCompYearly', 'isBrazil', 'LogComp']]
df_filtered


In [ ]:
sns.pairplot(data=df_filtered, height=3)
plt.show()

In [ ]:
sns.pairplot(data=df_filtered, height=3, hue='isBrazil', plot_kws={'alpha': 0.5}, diag_kind='kde')
plt.show()

In [ ]:
sns.pairplot(
    data=df_filtered,
    height=3,
    hue='isBrazil',
    palette='viridis',
    kind='hist',
    diag_kind='kde',
)
plt.show()



In [ ]:
sns.pairplot(
    data=df_filtered,
    height=3,
    hue='isBrazil',
    palette=['#acacac', '#34ba34'],
    hue_order=[0, 1],
    kind='hist',
    diag_kind='kde',
    diag_kws={'common_norm': False},
)
plt.show()

In [ ]:
#Jointplot -> cria histogramas e um gráfico de espalhamento, onde os histogramas mostram a distribuição para cada eixo

sns.jointplot(data=df_filtered, x='WorkExp', y='LogComp', height=8)
plt.show()

In [ ]:
sns.jointplot(
    data=df_filtered,
    x='WorkExp',
    y='LogComp',
    hue='isBrazil',
    height=8,
    kind='hist',
    marginal_kws={'kde': True},
)
plt.show()

In [ ]:
sns.jointplot(
    data=df_filtered,
    x='WorkExp',
    y='LogComp',
    hue='isBrazil',
    height=8,
    palette=['#acacac', '#34ba34'],
    hue_order=[0, 1],
    kind='kde',
    marginal_kws={'common_norm': False},
)
plt.show()

In [ ]:
g = sns.jointplot(
    data=df_filtered,
    x='WorkExp',
    y='LogComp',
    hue='isBrazil',
    height=8,
    palette=['#acacac', '#34ba34'],
    hue_order=[0, 1],
    kind='kde',
    marginal_kws={'common_norm': False},
)
g.plot_joint(sns.scatterplot, alpha=0.8, marker='*', s=100)
g.plot_marginals(sns.histplot, fill=False, stat='density', common_norm=False, alpha=0.5)
plt.show()

In [ ]:
#Cálculo de coeficientes de correlação


x = np.linspace(-100, 100, 1000)
y = x**5

corr, _ = pearsonr(x, y)
print(f'Correlação entre X e X^5: {corr:.3f}')


fig, ax = plt.subplots(figsize=(12, 4))

sns.lineplot(ax=ax, x=x, y=y)
ax.set_xlabel('x')
ax.set_ylabel('y')

fig.suptitle('Função $y = x^5$')
plt.show()

In [ ]:

x = np.linspace(-100, 100, 1000)
y = x**5

pearson_corr, _ = pearsonr(x, y)
print(f'Correlação de Pearson entre X e X^5: {pearson_corr:.3f}')
spearman_corr, _ = spearmanr(x, y)
print(f'Correlação de Spearman entre X e X^5: {spearman_corr:.3f}')

fig, ax = plt.subplots(figsize=(12, 4))

sns.lineplot(ax=ax, x=x, y=y)
ax.set_xlabel('x')
ax.set_ylabel('y')

fig.suptitle('Função $y = x^5$')
plt.show()

In [ ]:
df = pd.DataFrame({
    'X': [10, 20, 30, 40, 50],
    'Y': [1, 3, 4, 4, 3],
    'Z': [-10, -30, -33, -39, -200],
})

fig, (ax1, ax2, ax3) = plt.subplots(figsize=(18, 8), ncols=3)

sns.scatterplot(data=df, ax=ax1, x='X', y='Y', s=200)
ax1.set_title('X vs Y')
sns.scatterplot(data=df, ax=ax2, x='X', y='Z', s=200)
ax2.set_title('X vs Z')
sns.scatterplot(data=df, ax=ax3, x='Y', y='Z', s=200)
ax3.set_title('Y vs Z')

fig.suptitle('Gráficos de X, Y e Z')
plt.show()

In [ ]:
#Pearson - correlação padrão do pandas

fig, (ax1, ax2) = plt.subplots(figsize=(18, 8), ncols=2)

mask = np.triu(df.corr())

sns.heatmap(data=df.corr('pearson'), ax=ax1, annot=True, cbar=True, square=True, cmap='rocket', vmax=1.0, vmin=-1.0, mask=mask)
ax1.set_title('Correlação de Pearson')
sns.heatmap(data=df.corr('spearman'), ax=ax2, annot=True, cbar=True, square=True, cmap='rocket', vmax=1.0, vmin=-1.0, mask=mask)
ax2.set_title('Correlação de Spearman')

fig.suptitle('Comparação entre correlações')
plt.show()

In [ ]:
df = pd.read_csv(srp_path, index_col='ResponseId')
df_south_africa = df.loc[df['Country'] == 'South Africa'][['WorkExp', 'ConvertedCompYearly']].dropna().copy()
x = df_south_africa['WorkExp']
y = df_south_africa['ConvertedCompYearly']

regression = linregress(x, y)
regression

In [ ]:
print(f'Reta de regressão linear: y = {regression.slope:.2f}x + {regression.intercept:.2f}')
print(f'Coeficiente de correlação (Pearson): {regression.rvalue}')

In [ ]:
x_reg = np.linspace(0, 50, 100)
y_reg = x_reg * regression.slope + regression.intercept

fig, ax = plt.subplots(figsize=(12, 4))

sns.scatterplot(data=df_south_africa, ax=ax, x='WorkExp', y='ConvertedCompYearly')
sns.lineplot(ax=ax, x=x_reg, y=y_reg, color='#efab02')

fig.suptitle(f'Anos de experiência X salário - África do Sul (corr: {regression.rvalue:.2f})')
plt.show()

In [ ]:
# Filtro
df_south_africa_filtered = df_south_africa.loc[df_south_africa['ConvertedCompYearly'] < 200_000]

# Cálculo da regressão linear
x = df_south_africa_filtered['WorkExp']
y = df_south_africa_filtered['ConvertedCompYearly']
regression = linregress(x, y)

# Valores da reta de regressão
x_reg = np.linspace(0, 50, 100)
y_reg = x_reg * regression.slope + regression.intercept

# Gráfico
fig, ax = plt.subplots(figsize=(12, 4))

sns.scatterplot(data=df_south_africa_filtered, ax=ax, x='WorkExp', y='ConvertedCompYearly')
sns.lineplot(ax=ax, x=x_reg, y=y_reg, color='#efab02')

fig.suptitle(f'Anos de experiência X salário - África do Sul (corr: {regression.rvalue:.2f})')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

sns.regplot(data=df_south_africa_filtered, ax=ax, x='WorkExp', y='ConvertedCompYearly')

fig.suptitle(f'Anos de experiência X salário - África do Sul')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

sns.regplot(data=df_south_africa_filtered, ax=ax, x='WorkExp', y='ConvertedCompYearly', order=2)

fig.suptitle(f'Anos de experiência X salário - África do Sul')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

#0.01 foi usado para evitar erro com o fato de logarítmo dar erro com valor 0
sns.regplot(data=df_south_africa_filtered+0.01, ax=ax, x='WorkExp', y='ConvertedCompYearly', logx=True)

fig.suptitle(f'Anos de experiência X salário - África do Sul')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

sns.regplot(data=df_south_africa_filtered+0.01, ax=ax, x='WorkExp', y='ConvertedCompYearly', logistic=True)

fig.suptitle(f'Anos de experiência X salário - África do Sul')
plt.show()